# Enrichment Dataset Access Testing

This notebook tests access to each enrichment data source one at a time, using **Abbott's Booby** (*Papasula abbotti*) as the test species throughout — it is already confirmed in MongoDB so we have a known ground truth to check against.

**Data sources tested here:**
1. GBIF — species taxonomy lookup (already working in the pipeline)
2. GBIF — occurrence/observation data for migration patterns (new, replaces eBird)
3. IUCN Red List API v4 — conservation status
4. Avian Diet Database — local file (already working)
5. Wikipedia API — species description and habitat (new, no key needed)

**Instructions:** Run each section's cell(s) in order. After running all sections, fill in the summary table at the bottom.

---

In [1]:
# Import all the libraries needed across every section of this notebook.
# Run this cell FIRST before running any other cell.

import csv
import json
import os
import time
from collections import Counter
from pathlib import Path

import requests
from dotenv import load_dotenv

# Load API keys and credentials from the .env file.
# This makes IUCN_API_KEY and other secrets available without hardcoding them.
load_dotenv()

# Work out the project root folder so we can find the data/ directory.
# This works whether Jupyter was launched from the project root or from notebooks/.
PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root : {PROJECT_ROOT}")
print(f".env found   : {(PROJECT_ROOT / '.env').exists()}")

# The test species used in every section below.
TEST_LABEL = "ABBOTTS BOOBY"
TEST_SCIENTIFIC = "Papasula abbotti"

Project root : C:\workspace\neumont\AvianVision
.env found   : True


---
## Section 1 — GBIF: Species Taxonomy Lookup

GBIF (Global Biodiversity Information Facility) is a public API — no account or key needed.

This sends the scientific name to GBIF and gets back the full biological classification plus a **species key** (a numeric ID). That key is saved and reused in Section 2 to look up occurrence records.

In [2]:
# --- GBIF: Species taxonomy lookup ---
# Asks GBIF to match the scientific name to its internal species record.
# A successful response includes kingdom, phylum, class, order, family, genus,
# and a numeric species key we can use to look up sightings in Section 2.

GBIF_API = "https://api.gbif.org/v1"

url = f"{GBIF_API}/species/match"
params = {"name": TEST_SCIENTIFIC, "verbose": False}

response = requests.get(url, params=params, timeout=10)
print(f"Status code : {response.status_code}")

if response.status_code == 200:
    data = response.json()
    # Print the full response so we can see every field GBIF returns.
    print(json.dumps(data, indent=2))

    # Save the species key — we need it in Section 2 for the occurrence search.
    gbif_species_key = data.get("usageKey")
    print(f"\n>> GBIF species key (save for Section 2): {gbif_species_key}")
else:
    print(f"Error: {response.text}")
    gbif_species_key = None

Status code : 200
{
  "usageKey": 2480983,
  "scientificName": "Papasula abbotti (Ridgway, 1893)",
  "canonicalName": "Papasula abbotti",
  "rank": "SPECIES",
  "status": "ACCEPTED",
  "confidence": 99,
  "matchType": "EXACT",
  "kingdom": "Animalia",
  "phylum": "Chordata",
  "order": "Suliformes",
  "family": "Sulidae",
  "genus": "Papasula",
  "species": "Papasula abbotti",
  "kingdomKey": 1,
  "phylumKey": 44,
  "classKey": 212,
  "orderKey": 7192775,
  "familyKey": 9351,
  "genusKey": 2480982,
  "speciesKey": 2480983,
  "class": "Aves"
}

>> GBIF species key (save for Section 2): 2480983


---
## Section 2 — GBIF: Occurrence Data for Migration Patterns

The GBIF occurrence endpoint returns real bird sighting records submitted by observers worldwide. Each record includes the **country**, **coordinates**, and the **month** the bird was spotted.

By grouping sightings by month we can see:
- Which countries have sightings in which months (seasonal range)
- Where the breeding vs. wintering grounds appear to be

This is how we will get migration data instead of eBird.

> **Note:** Rare species like Abbott's Booby may have very few occurrence records — this is expected. The approach will work much better for common migratory species in the 525-species dataset.

In [3]:
# --- GBIF: Occurrence/observation records ---
# Fetches up to 300 sighting records for our test species.
# Each record includes a country code, coordinates (lat/lon), and the month observed.
# We tally these up by country and by month to show the seasonal distribution pattern.

if gbif_species_key is None:
    print("Skipping — no species key available. Run Section 1 first.")
else:
    url = f"{GBIF_API}/occurrence/search"
    params = {
        "speciesKey": gbif_species_key,
        "limit": 300,                   # GBIF allows a maximum of 300 per request
        "hasCoordinate": True,          # only include records that have lat/lon
        "hasGeospatialIssue": False,     # skip records with known location problems
    }

    response = requests.get(url, params=params, timeout=15)
    print(f"Status code : {response.status_code}")

    if response.status_code == 200:
        data = response.json()
        results = data.get("results", [])
        total_available = data.get("count", 0)

        print(f"Total records in GBIF for this species : {total_available}")
        print(f"Records returned this request          : {len(results)}")

        if results:
            # Count how many sightings were recorded in each country.
            country_counts = Counter()
            # Count how many sightings were recorded in each calendar month.
            month_counts = Counter()

            for record in results:
                country = record.get("countryCode") or "Unknown"
                month = record.get("month")
                country_counts[country] += 1
                if month:
                    month_counts[int(month)] += 1

            # Show the top countries where the species has been spotted.
            print("\n--- Sightings by country (top 10) ---")
            for country, count in country_counts.most_common(10):
                print(f"  {country:<6} : {count}")

            # Show the monthly breakdown as a simple bar chart.
            # A spike in certain months shows when the species is most active there.
            month_names = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun",
                           "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
            print("\n--- Sightings by month ---")
            for m in range(1, 13):
                count = month_counts.get(m, 0)
                bar = "#" * count
                print(f"  {month_names[m]} : {count:3d}  {bar}")

            # Print one full record so we can see what fields are available
            # and decide which ones to store in the pipeline.
            print("\n--- Example record (first result, all fields) ---")
            print(json.dumps(results[0], indent=2, default=str))
        else:
            print("No occurrence records found for this species in GBIF.")
    else:
        print(f"Error: {response.text}")

Status code : 200
Total records in GBIF for this species : 940
Records returned this request          : 300

--- Sightings by country (top 10) ---
  CX     : 253
  ID     : 20
  MP     : 20
  AU     : 7

--- Sightings by month ---
  Jan :  40  ########################################
  Feb :  11  ###########
  Mar :  13  #############
  Apr :  11  ###########
  May :  15  ###############
  Jun :   5  #####
  Jul :  23  #######################
  Aug :  24  ########################
  Sep :  23  #######################
  Oct :  11  ###########
  Nov :  71  #######################################################################
  Dec :  52  ####################################################

--- Example record (first result, all fields) ---
{
  "key": 6171005495,
  "datasetKey": "50c9509d-22c7-4a22-a47d-8c48425ef4a7",
  "publishingOrgKey": "28eb1a3f-1c15-4a95-931a-4af90ecb574d",
  "datasetCategory": [
    "CitizenScience"
  ],
  "installationKey": "997448a8-f762-11e1-a439-00145eb45e9a",


---
## Section 3 — IUCN Red List API v4

IUCN tracks the conservation status of every known species. We want:
- **Category** — e.g. EN (Endangered), LC (Least Concern), VU (Vulnerable)
- **Population trend** — increasing, stable, or decreasing

The v3 API was shut down in March 2025. We updated the pipeline to v4 in the last session but got a **401 Unauthorized** error. IUCN emailed instructions on how to update — if this cell still returns 401, compare the endpoint and headers below with what they told you and update this cell accordingly.

In [4]:
# --- IUCN Red List API v4 ---
# Two-step process:
#   Step 1 — look up the species by scientific name to get the latest assessment ID
#            and the Red List category code (e.g. EN = Endangered)
#   Step 2 — fetch the full assessment record using that ID to get the population trend
#
# Note: the [] fields in the Step 1 response (species_taxa, subpopulation_taxa, etc.)
# just mean this species has no recognized subspecies — that is completely normal.

IUCN_API = "https://api.iucnredlist.org/api/v4"

iucn_token = os.getenv("IUCN_API_KEY")

if not iucn_token:
    print("ERROR: IUCN_API_KEY not found in .env — check that the key is saved there.")
else:
    parts        = TEST_SCIENTIFIC.split()
    genus_name   = parts[0]
    species_name = parts[1]

    # --- Step 1: Species lookup ---
    # Returns the list of all historical assessments for this species.
    # We find the one marked latest=true and pull its assessment_id and category code.
    url     = f"{IUCN_API}/taxa/scientific_name"
    headers = {"Authorization": f"Bearer {iucn_token}"}
    params  = {
        "genus_name": genus_name,
        "species_name": species_name,
        "token": iucn_token,
    }

    print("Step 1 — species lookup")
    response = requests.get(url, headers=headers, params=params, timeout=10)
    print(f"Status : {response.status_code}")

    assessment_id = None
    category_code = None

    if response.status_code == 200:
        data = response.json()

        # Walk through the assessments list and find the most recent one (latest=True).
        # That is the current official Red List status for this species.
        for assessment in data.get("assessments", []):
            if assessment.get("latest"):
                assessment_id = assessment.get("assessment_id")
                category_code = assessment.get("red_list_category_code")
                break

        print(f"Category code  : {category_code}")
        print(f"Assessment ID  : {assessment_id}")
    else:
        print(f"Error: {response.text}")

    # --- Step 2: Full assessment details ---
    # The species lookup only returns the category code.
    # Population trend lives in the full assessment record, which needs a second request.
    if assessment_id:
        print(f"\nStep 2 — full assessment details (ID: {assessment_id})")
        detail_url = f"{IUCN_API}/assessment/{assessment_id}"

        detail_response = requests.get(detail_url, headers=headers,
                                       params={"token": iucn_token}, timeout=10)
        print(f"Status : {detail_response.status_code}")

        if detail_response.status_code == 200:
            # Print the full response so we can see every field that is available.
            detail = detail_response.json()
            print(json.dumps(detail, indent=2))
        else:
            print(f"Error: {detail_response.text}")
    else:
        print("\nSkipping Step 2 — no assessment_id was found in Step 1.")

Step 1 — species lookup
Status : 200
Category code  : EN
Assessment ID  : 152726109

Step 2 — full assessment details (ID: 152726109)
Status : 200
{
  "assessment_date": "2019-06-14T01:00:00.000+01:00",
  "year_published": "2019",
  "latest": true,
  "possibly_extinct": false,
  "possibly_extinct_in_the_wild": false,
  "sis_taxon_id": 22696649,
  "criteria": "B2ab(ii,iii)",
  "url": "https://www.iucnredlist.org/species/22696649/152726109",
  "citation": "BirdLife International 2019. Papasula abbotti. The IUCN Red List of Threatened Species 2019: e.T22696649A152726109. Accessed on 19 October 2026.",
  "assessment_id": 152726109,
  "assessment_points": false,
  "assessment_ranges": true,
  "taxon": {
    "sis_id": 22696649,
    "scientific_name": "Papasula abbotti",
    "species_taxa": [],
    "subpopulation_taxa": [],
    "infrarank_taxa": [],
    "kingdom_name": "ANIMALIA",
    "phylum_name": "CHORDATA",
    "class_name": "AVES",
    "order_name": "SULIFORMES",
    "family_name": "SULI

---
## Section 4 — Avian Diet Database (local file)

This database is already confirmed working in the pipeline. This is a quick check to make sure the file can still be found and read, and to look at what diet information exists for our test species.

> **Note:** The Avian Diet Database covers primarily North American birds. Abbott's Booby is a tropical Pacific species, so it may not appear in this file — that is expected, not a bug.

In [5]:
# --- Avian Diet Database ---
# Opens the local tab-separated diet file and searches for rows
# that match our test species by scientific name.
# Prints every matching row so we can see the full diet record.

DIET_FILE = PROJECT_ROOT / "data" / "AvianDietDatabase.txt"

print(f"File path   : {DIET_FILE}")
print(f"File exists : {DIET_FILE.exists()}")
print()

if DIET_FILE.exists():
    # Read through every row in the file and keep only the ones for our species.
    matching_rows = []
    with open(DIET_FILE, encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            if row.get("Scientific_Name", "").strip() == TEST_SCIENTIFIC:
                matching_rows.append(row)

    print(f"Rows found for '{TEST_SCIENTIFIC}' : {len(matching_rows)}")

    if matching_rows:
        # Print each matching row with all its fields.
        for i, row in enumerate(matching_rows, start=1):
            print(f"\n--- Row {i} ---")
            for key, value in row.items():
                print(f"  {key:<30} : {value}")
    else:
        print("No diet data found — expected for a non-North-American species.")
else:
    print("ERROR: Diet file not found. Check that PROJECT_ROOT printed correctly in the Imports cell.")

File path   : C:\workspace\neumont\AvianVision\data\AvianDietDatabase.txt
File exists : True

Rows found for 'Papasula abbotti' : 0
No diet data found — expected for a non-North-American species.


---
## Section 5 — Wikipedia API

Wikipedia has a detailed article for most bird species. The **MediaWiki REST API** is completely free — no account or key needed.

We will test two things:
1. **Search** — find the right Wikipedia article for the scientific name
2. **Summary + sections** — get the intro paragraph and list all section headings

The most useful fields for AvianVision will be:
- `description` — Wikipedia's one-line label for the species
- `extract` — plain-text intro paragraph (good for a dashboard species card)
- Section headings — tell us what structured information is in the article (Habitat, Diet, Behaviour, etc.)

In [6]:
# --- Wikipedia API: Search for the article ---
# Searches Wikipedia using the scientific name to find the correct article title.
# Searching by scientific name is more reliable than the common name for rare birds.
# We save the top result and use it in the next cell to fetch the full article data.

WIKI_API  = "https://en.wikipedia.org/w/api.php"
WIKI_REST = "https://en.wikipedia.org/api/rest_v1"

# Wikipedia blocks any request that does not include a descriptive User-Agent header.
# Without this, every request returns 403 Forbidden.
# The required format is: AppName/version (contact info)
WIKI_HEADERS = {
    "User-Agent": "AvianVision/1.0 (neumont capstone project; gjarrett@student.neumont.edu)"
}

search_params = {
    "action": "query",
    "list": "search",
    "srsearch": TEST_SCIENTIFIC,  # search using the scientific name
    "srlimit": 3,                  # return the top 3 results so we can check them
    "format": "json",
}

print(f"Searching Wikipedia for: {TEST_SCIENTIFIC}")
search_response = requests.get(WIKI_API, params=search_params, headers=WIKI_HEADERS, timeout=10)
print(f"Status code : {search_response.status_code}")

wiki_title = None

if search_response.status_code == 200:
    search_data = search_response.json()
    search_hits = search_data.get("query", {}).get("search", [])

    print("\nSearch results:")
    for hit in search_hits:
        print(f"  - {hit['title']}")

    # Use the first (most relevant) result as the article title.
    if search_hits:
        wiki_title = search_hits[0]["title"]
        print(f"\n>> Using article: '{wiki_title}'")
else:
    print(f"Error: {search_response.text}")

Searching Wikipedia for: Papasula abbotti
Status code : 200

Search results:
  - Abbott's booby
  - Sulidae
  - Suliformes

>> Using article: 'Abbott's booby'


In [7]:
# --- Wikipedia API: Fetch article summary and section list ---
# Uses the article title found in the cell above to fetch:
#   Part A — a plain-text summary (the intro paragraph)
#   Part B — the list of section headings inside the article
#
# WIKI_HEADERS (with the User-Agent) must be defined by running the search cell above first.

if wiki_title is None:
    print("Skipping — no article title found. Run the search cell above first.")
else:
    # --- Part A: Summary ---
    # The REST API summary endpoint returns a clean plain-text extract,
    # a one-sentence description, and a thumbnail image URL if available.
    summary_url = f"{WIKI_REST}/page/summary/{requests.utils.quote(wiki_title)}"
    summary_response = requests.get(summary_url, headers=WIKI_HEADERS, timeout=10)
    print(f"Summary status code : {summary_response.status_code}")

    if summary_response.status_code == 200:
        summary = summary_response.json()
        print(f"\nTitle       : {summary.get('title')}")
        print(f"Description : {summary.get('description')}")
        print(f"\nExtract (intro paragraph):")
        # The extract is the opening paragraph of the article in plain text.
        # This is the most useful field — it describes the species in 2-4 sentences.
        print(summary.get("extract", "No extract available."))
    else:
        print(f"Summary error: {summary_response.text}")

    print("\n" + "-" * 60)

    # --- Part B: Section list ---
    # The action API returns all the heading titles in the article.
    # This tells us what structured information is available
    # (e.g. Habitat, Diet, Breeding behaviour) so we can decide what to store.
    sections_params = {
        "action": "parse",
        "page": wiki_title,
        "prop": "sections",
        "format": "json",
    }
    sections_response = requests.get(WIKI_API, params=sections_params, headers=WIKI_HEADERS, timeout=10)
    print(f"\nSections status code : {sections_response.status_code}")

    if sections_response.status_code == 200:
        sections_data = sections_response.json()
        sections = sections_data.get("parse", {}).get("sections", [])
        print("\nArticle sections:")
        for section in sections:
            # Indent sub-sections to make the structure easier to read.
            indent = "  " * (int(section.get("toclevel", 1)) - 1)
            print(f"  {indent}{section.get('number', '')}. {section.get('line', '')}")
    else:
        print(f"Sections error: {sections_response.text}")

Summary status code : 200

Title       : Abbott's booby
Description : Species of bird

Extract (intro paragraph):
Abbott's booby is an endangered seabird of the sulid family, which includes gannets and boobies. It is a large booby and is placed within its own monotypic genus. It was first identified from a specimen collected by William Louis Abbott, who discovered it on Assumption Island in 1892.

------------------------------------------------------------

Sections status code : 200

Article sections:
  1. Taxonomy
  2. Description
  3. Distribution and habitat
  4. Population
  5. Ecology
    5.1. Breeding
  6. Threats and conservation
  7. References
  8. Further reading
  9. External links


---
## Results Summary

After running all sections above, fill in the table below with what each test returned.

| Dataset | Status | Key fields available | Notes |
|---|---|---|---|
| GBIF Taxonomy | Works | all expeted fields |  |
| GBIF Occurrences | Works | all expeted fields | |
| IUCN Red List v4 | works | Bird poulation and endager stats | |
| Avian Diet Database | Works | All expted filds | |
| Wikipedia API | work | All the sections I nned |  |

Once all datasets that work are confirmed, the next step is to update `src/enrich_pipeline.py` to include the new sources (GBIF occurrences, Wikipedia) and fix any that are still broken (IUCN).